# 📈 Project #11: Google (GOOGL) Stock Price Prediction
### 🏛️ Data Science Portfolio: 11 / 21

**Architect:** Kemal Demirbaş 🏰🚀
**Framework:** Equity Trend Analysis | Time Series Machine Learning

---

## 🎯 Project Objective
This project implements an advanced predictive model for **Alphabet Inc. (GOOGL)** stock prices. By integrating live market data from Yahoo Finance and engineering professional technical indicators (RSI, MA), we aim to forecast equity trends and price movements using a high-performance **Random Forest Regressor**.

---

## 🛠️ The 10-Step Engineering Discipline
1.  **Objective Definition:** Forecasting the next day's closing price for GOOGL equity.
2.  **EDA:** Analyzing historical volatility and long-term growth patterns.
3.  **Feature Selection:** Utilizing Open, High, Low, Close, and Volume (OHLCV) data.
4.  **Transformation:** Normalizing live data streams from the `yfinance` API.
5.  **Cleansing:** Eliminating gaps caused by market holidays and non-trading days.
6.  **Feature Engineering:** Architecting **RSI (Relative Strength Index)** and **7/30-day Moving Averages**.
7.  **Encoding:** Ensuring numerical consistency for regression analysis.
8.  **Partitioning:** Strictly **Chronological (Non-Shuffled)** train-test split (80/20).
9.  **Model Execution:** Training a robust **Random Forest Regressor** optimized for equity data.
10. **Performance Audit:** Evaluating accuracy through **R2 Score** and **RMSE**.

---



In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [3]:
# ==============================================================================
# --- STEP 1 & 2: Live Data Ingestion (Alphabet Inc.) ---
# ==============================================================================
ticker = "GOOGL"
# Fetching 5 years of historical data for robust training
df = yf.download(ticker, start="2020-01-01", end="2026-03-23")

# Flattening MultiIndex columns if present
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f"✅ {ticker} market data successfully synchronized.")

/tmp/ipykernel_150/3807844958.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2020-01-01", end="2026-03-23")
[*********************100%***********************]  1 of 1 completed

✅ GOOGL market data successfully synchronized.


In [6]:
# ==============================================================================
# --- STEP 3, 4 & 5: Feature Selection, Transformation & Cleansing ---
# ==============================================================================
# Selecting core OHLCV features and handling date transformation
df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
df.index = pd.to_datetime(df.index)

# Cleansing: Checking for missing trade days and ensuring data continuity
df.dropna(inplace=True)
print(f"✅ Steps 3, 4, 5 complete: {len(df)} trade days sanitized.")

✅ Steps 3, 4, 5 complete: 1562 trade days sanitized.


In [11]:
# --- RSI Helper Function (Required for Step 6) ---
def calculate_rsi(data, window=14):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

# --- STEP 6: Professional Feature Engineering (Returns Logic) ---
# We pivot to 'Percentage Change' to handle market stationarity
df['Returns'] = df['Close'].pct_change()

# Engineering Momentum & Trend Indicators based on Returns
df['MA7'] = df['Returns'].rolling(window=7).mean()
df['MA30'] = df['Returns'].rolling(window=30).mean()

# RSI remains a powerful momentum indicator for the underlying price
df['RSI'] = calculate_rsi(df['Close'])

# TARGET: Predicting the next day's percentage return
df['Target_Returns'] = df['Returns'].shift(-1)

# Cleansing: Removing NaNs generated by shifts and rolling windows
df.dropna(inplace=True)

print("✅ Feature Engineering (Returns-Based) Complete.")

✅ Feature Engineering (Returns-Based) Complete.


In [8]:
# ==============================================================================
# --- STEP 7: Encoding & Structural Alignment ---
# ==============================================================================
# Since stock data is numerical, we ensure all features are in float format
features = ['Close', 'Open', 'High', 'Low', 'Volume', 'MA7', 'MA30', 'RSI']
df[features] = df[features].astype(float)
print("✅ Step 7: Numerical encoding and structural alignment verified.")

✅ Step 7: Numerical encoding and structural alignment verified.


In [12]:
# --- STEP 8 & 9: Chronological Partitioning & Model Execution ---
# We define features that capture momentum and volatility
features = ['Returns', 'MA7', 'MA30', 'RSI', 'Volume']
X = df[features]
y = df['Target_Returns']

# Sequential Splitting (80% Train / 20% Test) to prevent data leakage
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# Initializing the Equity Forecasting Engine
# Increased n_estimators for better convergence on volatile returns
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, random_state=42)

In [13]:
# ==============================================================================
# --- STEP 10: Performance Audit ---
# ==============================================================================
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(f"\n=== 🏆 FINAL RESULTS ===\n📊 R2 Score: {r2:.4f}")

# Visual Validation
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_test.index, y=y_test, name='Actual', line=dict(color='gold')))
fig.add_trace(go.Scatter(x=y_test.index, y=y_pred, name='Predict', line=dict(color='cyan', dash='dot')))
fig.update_layout(title='Project #11: Google Equity Analysis', template='plotly_dark')
fig.show()


=== 🏆 FINAL RESULTS ===
📊 R2 Score: -0.0677


In [14]:
import joblib
from google.colab import files

# 1. Save the trained model to a file
# Note: Ensure 'model' is the name of your trained RandomForestRegressor
model_filename = 'google_model.joblib'
joblib.dump(model, model_filename)

print(f"✅ Google Equity Model sealed and saved as: {model_filename}")

# 2. Download the model to your local machine for Hugging Face deployment
files.download(model_filename)

✅ Google Equity Model sealed and saved as: google_model.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## 🏆 Final Conclusion & Quantitative Audit

The **Project #11: Google (GOOGL) Stock Price Prediction** serves as a critical case study in financial time-series analysis. By transitioning from raw price forecasting to **Daily Log-Returns**, the model was tested against the true volatility of the equity market.

### 📊 Strategic Architectural Insights
* **Market Entropy:** Stock market data is inherently **stochastic** (noisy). In this project, I prioritized the model's structural realism by attempting to forecast **daily returns** instead of raw prices. The fact that the performance score (R2) remains near the zero-baseline serves as empirical evidence of how "efficient" and unpredictable the market truly is.
* **Efficient Market Hypothesis (EMH):** The resulting near-zero R2 score confirms that daily returns in highly liquid assets like Alphabet Inc. behave as a random walk, where historical patterns are rapidly priced in by the market.

### 🚀 Technical Takeaway & Live Link
This project successfully establishes a foundational understanding of risk-adjusted modeling and data stationarity in high-entropy systems.

👉 **[Live Google Equity Forecaster on Hugging Face](https://huggingface.co/spaces/Ironside35/google-stock-predictor)** 📈🚀

---
**Architect:** Kemal Demirbaş 🏰🚀  
**Project #11 of 21** | *Mastering the volatility of financial intelligence.*